In [1]:
import pandas as pd
import os

output_dir = '..SentinelNet/data/processed/'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
save_path = os.path.join(output_dir, 'train_processed_advanced.csv')
df = pd.read_csv(save_path)

In [2]:
df

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH,attack_class
0,0.000000,1.619565,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,normal
1,0.000000,0.369565,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,normal
2,0.000000,-0.159420,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,-1.000000,0.0,DoS
3,0.000000,0.681159,15.800388,0.0,0.0,0.0,0.000000,0.00000,1.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,normal
4,0.000000,0.561594,0.813953,0.0,0.0,0.0,0.000000,0.00000,1.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336710,0.000000,-0.159420,4.015504,0.0,0.0,0.0,1.000000,0.00000,1.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,U2R
336711,0.000000,-0.159420,8.532773,0.0,0.0,0.0,0.355746,0.00000,1.0,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,U2R
336712,297.076270,2.246518,106.318435,0.0,0.0,0.0,2.315200,0.00000,1.0,3.086933,...,0.0,0.228267,0.0,0.0,0.0,0.0,0.0,-0.228267,0.0,U2R
336713,165.025534,5.512957,5.346556,0.0,0.0,0.0,3.000000,0.00000,1.0,4.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,U2R


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
# Select the features you want to use for training
features = ['duration', 'src_bytes', 'dst_bytes'] 
X = df[features]
X

,duration,src_bytes,dst_bytes
0,0.000000,1.619565,0.000000
1,0.000000,0.369565,0.000000
2,0.000000,-0.159420,0.000000
3,0.000000,0.681159,15.800388
4,0.000000,0.561594,0.813953
...,...,...,...
336710,0.000000,-0.159420,4.015504
336711,0.000000,-0.159420,8.532773
336712,297.076270,2.246518,106.318435
336713,165.025534,5.512957,5.346556


In [4]:
model = IsolationForest(n_estimators=100, max_samples='auto', contamination=0.01, random_state=42)
model.fit(X)

,n_estimators,100
,max_samples,'auto'
,contamination,0.01
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [5]:
df['anomaly'] = model.predict(X)

In [6]:
df['anomaly_score'] = model.decision_function(X)

In [7]:
anomalies = df.loc[df['anomaly'] == -1]

print(f"Number of anomalies detected: {len(anomalies)}")
print("Detected anomalies (potential intrusions):")
print(anomalies[features + ['anomaly_score']])


Number of anomalies detected: 3116
Detected anomalies (potential intrusions):
            duration     src_bytes   dst_bytes  anomaly_score
303      5043.000000  1.859196e+04    0.000000      -0.004634
1038    35682.000000  1.383004e+06    0.000000      -0.054776
3341     3995.000000  1.778841e+03  133.139535      -0.025479
5261     5066.000000  1.860084e+04    0.000000      -0.002056
5930    16800.000000  6.978261e+00  196.290698      -0.010349
...              ...           ...         ...            ...
269339   5063.269780  1.860084e+04    0.000000      -0.003086
269340   5061.523776  1.860084e+04    0.000000      -0.001027
269364   5041.000000  1.860355e+04    0.000000      -0.001950
269379   5062.364834  1.860084e+04    0.000000      -0.001542
269414   5065.271501  1.860084e+04    0.000000      -0.002056

[3116 rows x 4 columns]


In [ ]:
df